# 01 — Quantum Computing Basics (Qiskit)
### QML Curriculum · Companion Notebook to Module 5 — *Quantum Computing Foundations*

**Part of:** [Schrödinger's Cats — Quantum Machine Learning Platform](https://tanjinadnanabir.github.io/WISER-Education-Challenge/)

This notebook is the hands-on partner to Module 5. Everything you saw as sliders and Bloch spheres in the browser tutorial, you'll now build with real code, using **Qiskit** — IBM's quantum computing SDK.

**By the end of this notebook you will be able to:**
- Build single- and multi-qubit circuits in Qiskit
- Apply gates (X, Z, H, RX, RY, RZ, CNOT) and read the resulting statevector
- Visualize a qubit's state on the Bloch sphere
- Simulate measurement and the Born rule
- Build your first entangled Bell pair

**Prerequisites:** Module 4 & 5 (conceptual), or the Prep Primer if you're new to linear algebra / complex numbers.

> 💡 Every code cell is meant to be run in order — variables carry over between cells.


In [1]:
# Setup — run this first
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_bloch_multivector, plot_bloch_vector, plot_histogram
from qiskit_aer import AerSimulator

np.set_printoptions(precision=3, suppress=True)
print("Qiskit environment ready.")

Qiskit environment ready.


## 1. A Qubit is a Vector

Just like Module 5, Chapter 1 showed: a qubit's state is a column vector `[amplitude_for_0, amplitude_for_1]`.

In Qiskit, a fresh qubit always starts in the state **|0⟩** = `[1, 0]`. Let's build a 1-qubit circuit and inspect its statevector directly — no gates yet.

In [2]:
qc = QuantumCircuit(1)
state = Statevector.from_instruction(qc)
print("State vector:", state.data)
print("Probabilities:", state.probabilities())

State vector: [1.+0.j 0.+0.j]
Probabilities: [1. 0.]


## 2. Applying Gates = Multiplying by a Matrix

Recall from Module 5, Chapter 4: a gate is a **unitary matrix**. Applying it to a qubit is matrix-vector multiplication. Let's apply the **X gate** (bit-flip) and watch |0⟩ become |1⟩.

In [3]:
qc = QuantumCircuit(1)
qc.x(0)               # apply the X gate
print(qc.draw())

state = Statevector.from_instruction(qc)
print("\nState vector:", state.data)
print("Probabilities:", state.probabilities())

   ┌───┐
q: ┤ X ├
   └───┘

State vector: [0.+0.j 1.+0.j]
Probabilities: [0. 1.]


Now let's apply the **Hadamard gate (H)** — the gate that creates superposition. Starting from |0⟩, H should land us exactly on |+⟩ = `[0.707, 0.707]`, a 50/50 mix of |0⟩ and |1⟩.

In [4]:
qc = QuantumCircuit(1)
qc.h(0)
print(qc.draw())

state = Statevector.from_instruction(qc)
print("\nState vector:", state.data)
print("Probabilities:", state.probabilities())

   ┌───┐
q: ┤ H ├
   └───┘

State vector: [0.707+0.j 0.707+0.j]
Probabilities: [0.5 0.5]


### 🔧 Try it yourself
Change the cell below to build `|−⟩` instead of `|+⟩`. *Hint: apply `X` then `H`, or `H` then `Z` — try both and compare.*

In [5]:
qc = QuantumCircuit(1)
# YOUR CODE HERE — try qc.x(0); qc.h(0)  OR  qc.h(0); qc.z(0)


state = Statevector.from_instruction(qc)
print("State vector:", state.data)

State vector: [1.+0.j 0.+0.j]


## 3. Visualizing on the Bloch Sphere

Module 5, Chapter 6 gave you sliders for θ and φ. Here's the real Bloch sphere, generated directly from a circuit's statevector.

In [6]:
def bloch_vector(statevector):
    # Convert a 1-qubit statevector [a, b] into Bloch (x, y, z) coordinates
    a, b = statevector.data
    x = 2 * np.real(np.conj(a) * b)
    y = 2 * np.imag(np.conj(a) * b)
    z = (np.abs(a) ** 2) - (np.abs(b) ** 2)
    return [x, y, z]

circuits = {"|0>": QuantumCircuit(1), "H -> |+>": QuantumCircuit(1)}
circuits["H -> |+>"].h(0)

for name, c in circuits.items():
    sv = Statevector.from_instruction(c)
    vec = bloch_vector(sv)
    print(f"{name}: Bloch vector = [{vec[0]:.2f}, {vec[1]:.2f}, {vec[2]:.2f}]")
    fig = plot_bloch_vector(vec, title=name, figsize=(4, 4))
    plt.show()

|0>: Bloch vector = [0.00, 0.00, 1.00]
H -> |+>: Bloch vector = [1.00, 0.00, 0.00]


## 4. Measurement — The Born Rule in Practice

Module 5, Chapter 7: `P(outcome) = |⟨outcome|ψ⟩|²`. Instead of computing this by hand, we now **sample** it — run the circuit thousands of times on a simulator and look at the histogram of outcomes.

In [7]:
qc = QuantumCircuit(1, 1)
qc.h(0)            # put qubit into |+> superposition
qc.measure(0, 0)   # measure it

simulator = AerSimulator()
result = simulator.run(qc, shots=2000).result()
counts = result.get_counts()

print("Measured counts over 2000 shots:", counts)
plot_histogram(counts)
plt.show()

Measured counts over 2000 shots: {'0': 962, '1': 1038}


Notice the counts split roughly 50/50 between `0` and `1` — exactly the probability computed by the Born rule for |+⟩, converging as you saw in the Module 5 measurement demo.

## 5. Multi-Qubit Circuits & Entanglement

A 2-qubit register needs a 4-element vector: `[amp(00), amp(01), amp(10), amp(11)]`. Let's build the most famous entangled state in quantum computing: the **Bell pair**.

Recipe: `H` on qubit 0, then `CNOT` (controlled-X) from qubit 0 to qubit 1.

In [8]:
qc = QuantumCircuit(2, 2)
qc.h(0)
qc.cx(0, 1)   # CNOT: flips qubit 1 only if qubit 0 is |1>
print(qc.draw())

state = Statevector.from_instruction(qc)
print("\nState vector [00, 01, 10, 11]:", state.data)
print("Probabilities:", dict(zip(["00","01","10","11"], state.probabilities())))

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
c: 2/══════════
               

State vector [00, 01, 10, 11]: [0.707+0.j 0.   +0.j 0.   +0.j 0.707+0.j]
Probabilities: {'00': np.float64(0.4999999999999999), '01': np.float64(0.0), '10': np.float64(0.0), '11': np.float64(0.4999999999999999)}


Notice: only `00` and `11` have nonzero probability. The qubits are **entangled** — exactly like Module 4, Chapter 5's "create a Bell pair & measure" demo. Measuring qubit 0 instantly tells you what qubit 1 will be.

Let's confirm with real repeated measurement.

In [9]:
qc.measure([0, 1], [0, 1])
result = simulator.run(qc, shots=2000).result()
counts = result.get_counts()
print("Counts:", counts)
plot_histogram(counts)
plt.show()

print("\nNotice: only '00' and '11' ever appear — never '01' or '10'.")
print("The two qubits are perfectly correlated, exactly like the entanglement demo in Module 4.")

Counts: {'00': 1026, '11': 974}



Notice: only '00' and '11' ever appear — never '01' or '10'.
The two qubits are perfectly correlated, exactly like the entanglement demo in Module 4.


## 6. Recap & What's Next

| Module 5 Chapter | What you just did in code |
|---|---|
| Linear Algebra Basics | Inspected raw statevectors as NumPy arrays |
| Quantum Gates | Applied X, H as literal unitary matrices |
| Bloch Sphere | Rendered real θ/φ positions with `plot_bloch_multivector` |
| Quantum Measurement | Sampled the Born rule with `AerSimulator` |
| *(Module 4)* Entanglement | Built and measured a real Bell pair |

**Next notebook: `02_quantum_data_encoding.ipynb`** — Module 8. You'll learn how to turn ordinary classical data (numbers, images, features) into quantum states — the very first step of every QML pipeline.
